In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import requests
import time
import json
import warnings
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, classification_report,
                             confusion_matrix)
from sklearn.impute import SimpleImputer
warnings.filterwarnings("ignore")

print("All imports successful")

All imports successful


In [ ]:
url = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"
# https://sdmdataaccess.sc.egov.usda.gov/WebServiceHelp.aspx#SDMTabularService

def query_soil(sql, label=""):
    response = requests.post(
        url,
        data={"query": sql, "format": "JSON+COLUMNNAME"},
        timeout=30
    )
    if response.status_code == 200:
        data = response.json()
        # Convert to DataFrame
        cols = data["Table"][0]
        rows = data["Table"][1:]
        df = pd.DataFrame(rows, columns=cols)
        print(f"{label} — {len(df)} rows returned")
        return df
    else:
        print(f"Error {response.status_code}:", response.text)
        return None

In [ ]:
# https://sdmdataaccess.sc.egov.usda.gov/documents/TableColumnDescriptionsReport.pdf

texas_features = query_soil("""
    SELECT 
        -- 1. Basic Information
        l.areasymbol, 
        m.musym, 
        m.muname,
        c.compname,
        s.saverest,
        
        -- 2. Soil Features (Category/Attribute)
        c.taxorder,        -- Soil Type (Order)
        c.drainagecl,      -- Drainage Class (e.g., Well drained)
        c.elev_r,          -- Elevation (Representative)
        c.slope_r,         -- Slope (Representative)
        
        -- 3. Soil Horizon Numerical Data (from chorizon table)
        ch.hzdept_r,       -- Top depth of horizon
        ch.hzdepb_r,       -- Bottom depth of horizon (Depth to restrictive layer)
        ch.ph1to1h2o_r,    -- pH (1:1 Water)
        ch.om_r,           -- Organic Matter (Proxy for N-P-K)
        ch.ec_r,           -- Electrical Conductivity (Salinity)
        ch.cec7_r,         -- Cation Exchange Capacity (Nutrient storage)
        ch.awc_r,          -- Available Water Capacity (Water storage)
                                
        -- 4. Crop data
        cp.cropname,       -- Crop type (e.g., Corn, Soybeans, Cotton)
        cp.yldunits,       -- Crop yield units per area
        cp.nonirryield_r,   -- The expected yield per acre without supplemental irrigation.
        cp.irryield_r      -- The expected yield per acre with irrigation.
                      

    FROM legend l
    INNER JOIN sacatalog s ON s.areasymbol = l.areasymbol
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    INNER JOIN chorizon ch ON ch.cokey = c.cokey
    LEFT OUTER JOIN cocropyld cp ON cp.cokey = c.cokey

    WHERE l.areasymbol LIKE 'TX%'
    AND c.majcompflag = 'Yes'
    AND ch.hzdept_r = 0 -- Wanna see the soil where the root is in
""","texas")


# ─────────────────────────────────────────
# 4. Save to CSV
# ─────────────────────────────────────────
for df, name in [
    (texas_features, "texas"),
]:
    if df is not None:
        df.to_csv(f"{name}.csv", index=False)
        print(f"Saved {name}.csv")

❌ Error 400: <?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.1.1/OGC-exception.xsd">
<ServiceException>
Maximum allowable number of returned records (100000) exceeded.</ServiceException>
</ServiceExceptionReport>



In [3]:
stations_df = pd.read_fwf("../data/ghcnd-stations.txt", header=None, 
                          widths=[11, 9, 10, 7, 3, 31, 3], 
                          names=['ID', 'lat', 'lon', 'elev', 'state', 'name', 'gsn'])

print(stations_df)

                 ID      lat      lon    elev state                   name  \
0       ACW00011604  17.1167 -61.7833    10.1   NaN  ST JOHNS COOLIDGE FLD   
1       ACW00011647  17.1333 -61.7833    19.2   NaN               ST JOHNS   
2       AE000041196  25.3330  55.5170    34.0   NaN    SHARJAH INTER. AIRP   
3       AEM00041194  25.2550  55.3640    10.4   NaN             DUBAI INTL   
4       AEM00041217  24.4330  54.6510    26.8   NaN         ABU DHABI INTL   
...             ...      ...      ...     ...   ...                    ...   
129652  ZI000067969 -21.0500  29.3670   861.0   NaN         WEST NICHOLSON   
129653  ZI000067975 -20.0670  30.8670  1095.0   NaN               MASVINGO   
129654  ZI000067977 -21.0170  31.5830   430.0   NaN          BUFFALO RANGE   
129655  ZI000067983 -20.2000  32.6160  1132.0   NaN               CHIPINGE   
129656  ZI000067991 -22.2170  30.0000   457.0   NaN             BEITBRIDGE   

        gsn  
0       NaN  
1       NaN  
2        GS  
3      

In [4]:
stations_df_tx = stations_df[stations_df['state'] == 'TX']
print(stations_df_tx)

                 ID      lat       lon   elev state                   name  \
92429   US1TXAC0002  33.8281  -98.5492  305.7    TX  WICHITA FALLS 5.2 SSW   
92430   US1TXAC0003  33.5838  -98.6298  323.7    TX    ARCHER CITY 0.7 SSW   
92431   US1TXAC0005  33.7762  -98.5350  300.2    TX    WICHITA FALLS 8.5 S   
92432   US1TXAC0009  33.8328  -98.5360  300.5    TX  WICHITA FALLS 4.6 SSW   
92433   US1TXAD0002  32.0533 -102.8796  973.5    TX        ANDREWS 26.7 SW   
...             ...      ...       ...    ...   ...                    ...   
128639  USW00093955  33.6333  -95.4500  166.7    TX                  PARIS   
128645  USW00093983  31.7797  -95.7064  128.9    TX      PALESTINE MUNI AP   
128646  USW00093984  31.1500  -97.4167  207.9    TX   TEMPLE DRAUGHTON MIL   
128647  USW00093985  32.7817  -98.0603  287.1    TX       MINERAL WELLS AP   
128649  USW00093987  31.2358  -94.7547   87.2    TX  LUFKIN ANGELINA CO AP   

        gsn  
92429   NaN  
92430   NaN  
92431   NaN  
92432  

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def fetch_fips(row):
    id, lat, lon = row['ID'], row['lat'], row['lon']
    url = f"https://geo.fcc.gov/api/census/area?lat={lat}&lon={lon}&format=json"
    response = requests.get(url).json()
    if not response['results']:
        return None
    fips_code = response['results'][0]['county_fips']
    return {'ID': id, 'lat': lat, 'lon': lon, 'fips_code': fips_code}

results = []
rows = [row for _, row in stations_df_tx.iterrows()]

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(fetch_fips, row): row for row in rows}
    for future in tqdm(as_completed(futures), total=len(rows)):
        result = future.result()
        if result:
            results.append(result)

df_fips = pd.DataFrame(results)
print(df_fips.head())

100%|██████████| 6472/6472 [11:41<00:00,  9.23it/s]

            ID      lat      lon fips_code
0  US1TXAG0004  31.2058 -94.3357     48005
1  US1TXAG0001  31.3213 -94.8445     48005
2  US1TXAG0002  31.1464 -94.3856     48005
3  US1TXAC0005  33.7762 -98.5350     48009
4  US1TXAG0003  31.3034 -94.7627     48005


In [6]:
combined_df = pd.merge(stations_df_tx, df_fips, on='ID')
combined_df.head()

,ID,lat_x,lon_x,elev,state,name,gsn,lat_y,lon_y,fips_code
0,US1TXAC0002,33.8281,-98.5492,305.7,TX,WICHITA FALLS 5.2 SSW,NaN,33.8281,-98.5492,48009
1,US1TXAC0003,33.5838,-98.6298,323.7,TX,ARCHER CITY 0.7 SSW,NaN,33.5838,-98.6298,48009
2,US1TXAC0005,33.7762,-98.5350,300.2,TX,WICHITA FALLS 8.5 S,NaN,33.7762,-98.5350,48009
3,US1TXAC0009,33.8328,-98.5360,300.5,TX,WICHITA FALLS 4.6 SSW,NaN,33.8328,-98.5360,48009
4,US1TXAD0002,32.0533,-102.8796,973.5,TX,ANDREWS 26.7 SW,NaN,32.0533,-102.8796,48495


In [10]:
# Get list of Station IDs located in Texas (TX)
texas_station_ids = combined_df['ID'].unique()

# 2. Process the 2025 massive CSV with the TX filter
url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/2025.csv.gz"
cols = ['ID', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time']
target_elements = ['PRCP', 'TMAX', 'TMIN']

df_chunk = pd.read_csv(url, compression='gzip', names=cols, chunksize=200000, low_memory=False)

texas_data_list = []

print("Filtering 2025 data for Texas stations only...")

for chunk in df_chunk:
    # Filter by BOTH: 1. Element Type AND 2. Texas Station IDs
    is_target_element = chunk['element'].isin(target_elements)
    is_texas_ID = chunk['ID'].isin(texas_station_ids)
    
    target = chunk[is_target_element & is_texas_ID].copy()
    texas_data_list.append(target)


# Final Concatenation
texas_2025 = pd.concat(texas_data_list)

# Merge fips_code from combined_df by ID
texas_2025 = texas_2025.merge(combined_df[['ID', 'fips_code']], on='ID', how='left')

print(f"Finished! Final record count for Texas: {len(texas_2025)}")
print(texas_2025)

Filtering 2025 data for Texas stations only...
Finished! Final record count for Texas: 921755
                 ID      date element  value m_flag q_flag s_flag  obs_time  \
0       USW00023091  20250101    TMAX    139    NaN    NaN      W       NaN   
1       USW00023091  20250101    TMIN     -5    NaN    NaN      W       NaN   
2       USW00023091  20250101    PRCP      0    NaN    NaN      W       NaN   
3       USW00023906  20250101    TMAX    187    NaN    NaN      R       NaN   
4       USW00023906  20250101    TMIN    106    NaN    NaN      R       NaN   
...             ...       ...     ...    ...    ...    ...    ...       ...   
921750  US1TXWT0030  20251231    PRCP      0    NaN    NaN      N     700.0   
921751  US1TXWT0031  20251231    PRCP      0    NaN    NaN      N     700.0   
921752  US1TXWT0032  20251231    PRCP      0    NaN    NaN      N     900.0   
921753  US1TXYK0001  20251231    PRCP      0    NaN    NaN      N     700.0   
921754  US1TXZP0002  20251231    PRCP

In [ ]:
texas_2025['date'] = pd.to_datetime(texas_2025['date'], format='%Y%m%d')
texas_2025['month'] = texas_2025['date'].dt.month

fips_pivot = texas_2025.pivot_table(
    index=['fips_code', 'month'],
    columns='element',
    values='value',
    aggfunc='mean'
).reset_index

fips_pivot.columns.name = None

# 3. Unit Adjustment and Scaling
weather_texas_2025 = pd.DataFrame()
weather_texas_2025['fips_code'] = fips_pivot['fips_code']
weather_texas_2025['month'] = fips_pivot['month']
weather_texas_2025['precip_total_mm'] = fips_pivot['PRCP'] / 10
weather_texas_2025['tmax_avg_c'] = fips_pivot['TMAX'] / 10
weather_texas_2025['tmin_avg_c'] = fips_pivot['TMIN'] / 10

# 4. Export to CSV
weather_texas_2025.to_csv("texas_weather_2025.csv")

print("--- Monthly Weather Summary 2025 with Month Column ---")
print(weather_texas_2025)

--- Monthly Weather Summary 2025 with Month Column ---
    fips_code  precip_total_mm  tmax_avg_c  tmin_avg_c
0       48001         3.307340   26.099079   14.081952
1       48003         4.621951   26.045152   11.047922
2       48005         4.938741   26.826075   14.510680
3       48007         2.005004   28.065370   18.660166
4       48009         2.173699         NaN         NaN
..        ...              ...         ...         ...
239     48499         3.385784   24.107018   11.930113
240     48501         0.983090   25.382466    9.186575
241     48503         1.963918   26.043733   11.969081
242     48505         1.680172         NaN         NaN
243     48507         1.386798         NaN         NaN

[244 rows x 4 columns]
